# Домашнее задание: Бинарная классификация токсичности текстов

## Практический кейс

Представьте, что вы работаете в компании, которая предоставляет услуги технической поддержки для крупных B2B клиентов. Ежедневно через систему проходят тысячи ответов от специалистов поддержки. Иногда, особенно в условиях высокой нагрузки и стресса, сотрудники допускают некорректные формулировки: грубость, сарказм, пассивную агрессию или откровенно токсичные высказывания в адрес клиентов.

Такие инциденты несут прямые репутационные потери для бизнеса. Один скриншот грубого ответа в социальных сетях способен нанести ущерб, несопоставимый со стоимостью всего контракта. Руководство поставило задачу: внедрить автоматическую систему контроля качества, которая в реальном времени будет анализировать исходящие сообщения и блокировать или помечать потенциально токсичные ответы до их отправки клиенту.

Ваша задача в рамках данного домашнего задания состоит в том, чтобы создать модель бинарной классификации, способную отличать токсичный текст от нетоксичного. Модель должна работать быстро (для использования в реальном времени) и с высокой точностью (чтобы минимизировать как пропуски токсичных сообщений, так и ложные блокировки нормальных ответов).

## Рекомендации по выполнению в Google Colab

Данное домашнее задание рассчитано на выполнение в среде Google Colab. Ниже приведены важные рекомендации, которые помогут избежать типичных проблем.

**Выбор среды выполнения.** Перед началом работы переключите Runtime на GPU. Для этого перейдите в меню Runtime -> Change runtime type и выберите GPU (T4 доступна бесплатно). Без GPU дообучение модели займет неприемлемо долгое время.

**Если не хватает видеопамяти (CUDA Out of Memory).** Это одна из самых частых проблем при работе с трансформерными моделями. Вот что можно предпринять:

1. Уменьшите BATCH_SIZE. Начните с 64 или даже 32 вместо 128. Это самый простой и эффективный способ снизить потребление VRAM.
2. Уменьшите INPUT_MAX_TOKENS. Вместо 128 попробуйте 64. Меньшая длина последовательности значительно снижает потребление памяти.
3. Уменьшите размер датасета через параметр MAX_DATASET_LEN. Меньше данных означает меньше батчей и меньше нагрузки на память.
4. Используйте меньшую модель. Вместо large-версии модели попробуйте base-версию той же архитектуры.
5. Перезапустите Runtime. Иногда память остается занятой от предыдущих запусков. Перейдите в Runtime -> Restart runtime.
6. Используйте `torch.cuda.empty_cache()` и `gc.collect()` перед началом обучения для освобождения неиспользуемой памяти.

**Сохранение промежуточных результатов.** Colab может отключить сессию при длительном бездействии. Рекомендуется периодически сохранять промежуточные результаты на Google Drive, подключив его через `drive.mount('/content/drive')`.

**Установка зависимостей.** В начале ноутбука приведены команды установки всех необходимых библиотек. Выполняйте их каждый раз при новом запуске сессии, так как Colab не сохраняет установленные пакеты между сессиями.

## Распределение баллов

| Блок | Задание | Баллы |
|------|---------|-------|
| 0 | Формализация задачи | 5 |
| 1 | Сбор данных (положительный класс) | 5 |
| 2 | Очистка датасета | 5 |
| 3 | Разделение на подвыборки и балансировка | 5 |
| 4 | Количественная оценка и визуализация датасета | 5 |
| 5 | Публикация датасета на Hugging Face | 5 |
| 6 | Дообучение модели (код предоставлен) | 0 |
| 7 | Оценка модели и визуализация ошибок | 10 |
| 8 | Ручной инференс и замер времени GPU vs CPU | 5 |
| 9 | Публикация модели на Hugging Face | 5 |
| **Итого** | | **50** |

---

## Блок 0. Формализация задачи (5 баллов)

### 0.1 Формулировка бизнес-задачи

Опишите бизнес-задачу своими словами. Какую проблему мы решаем? Какие последствия несет нерешенная проблема для бизнеса? Какой результат ожидает заказчик?


In [ ]:
# Ваш ответ (в виде текста в markdown-ячейке выше или в комментариях здесь):
# ...


### 0.2 Сведение к ML-задаче

Заполните следующие пункты:


In [ ]:
# Тип задачи: ...
# Объект классификации: ...
# Класс 1 (токсичный): ...
# Класс 0 (нетоксичный): ...
# Целевая переменная: ...


### 0.3 Выбор ML-метрик

Обоснуйте выбор метрик для оценки качества модели. Подумайте: что важнее в данном бизнес-кейсе, Precision или Recall? Почему? Какие метрики вы будете использовать и зачем?


In [ ]:
# Ваш ответ:
# Основная метрика: ...
# Дополнительные метрики: ...
# Обоснование: ...


---

## Блок 1. Сбор данных (5 баллов)

### 1.1 Установка зависимостей


In [ ]:
!pip install -q pytorch-lightning transformers datasets torchmetrics huggingface_hub scikit-learn matplotlib seaborn pandas numpy


### 1.2 Импорт библиотек


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')


### 1.3 Загрузка токсичного класса (Класс 1) — код заполнен

Ниже приведен код загрузки данных для токсичного класса. Эти данные уже подготовлены и не требуют модификации с вашей стороны. Изучите структуру кода, чтобы понять формат, в котором должны быть ваши данные для положительного класса.


In [ ]:
df = pd.DataFrame(columns=['text', 'label'])

# Источник 1: Русскоязычный датасет токсичных комментариев
if True:
    toxic_df = pd.read_csv("https://raw.githubusercontent.com/skoltech-nlp/russian_toxicity_classifier/master/data/train.csv")
    toxic_df = toxic_df[toxic_df['toxic'] == 1][['comment']]
    toxic_df = toxic_df.rename(columns={'comment': 'text'})
    toxic_df['label'] = 1
    df = pd.concat([df, toxic_df], ignore_index=True)

# Источник 2: Датасет оскорбительных высказываний
if True:
    try:
        offensive_df = pd.read_csv("https://raw.githubusercontent.com/skoltech-nlp/russian_toxicity_classifier/master/data/test.csv")
        offensive_df = offensive_df[offensive_df['toxic'] == 1][['comment']]
        offensive_df = offensive_df.rename(columns={'comment': 'text'})
        offensive_df['label'] = 1
        df = pd.concat([df, offensive_df], ignore_index=True)
    except:
        print("Не удалось загрузить дополнительный источник токсичных данных")

print(f"Загружено записей токсичного класса: {len(df)}")
df.sample(5)


### 1.4 Загрузка нетоксичного класса (Класс 0) — задание для студента

Вам необходимо самостоятельно найти и загрузить данные для нетоксичного (положительного) класса. Итоговый DataFrame должен содержать колонки `text` и `label` (где label = 0).

**Подсказки по поиску данных:**

- Подумайте, какие тексты гарантированно не содержат токсичности. Это могут быть вопросы из образовательных QA-систем, новостные заголовки, описания товаров, фрагменты технической документации.
- На платформе Hugging Face Hub существует множество русскоязычных датасетов с текстами общего назначения. Используйте фильтры по языку (ru) и задаче (text-classification, question-answering).
- Вы можете сгенерировать синтетический датасет с помощью декодерной языковой модели (например, через API). Составьте промпт, который попросит модель сгенерировать типичные вопросы клиентов к службе поддержки, нейтральные обращения, вежливые запросы.
- Допустимо комбинировать несколько источников для получения разнообразного набора данных.
- Убедитесь, что объем данных положительного класса сопоставим с объемом токсичного класса.


In [ ]:
# === ВАШ КОД ЗДЕСЬ ===
# Загрузите данные для нетоксичного класса (label = 0)
# Итоговый формат: DataFrame с колонками 'text' и 'label'

# Пример структуры:
# class0_df = pd.read_...("ваш_источник")
# class0_df = class0_df[['нужная_колонка']]
# class0_df = class0_df.rename(columns={'нужная_колонка': 'text'})
# class0_df['label'] = 0
# df = pd.concat([df, class0_df], ignore_index=True)

# === КОНЕЦ ВАШЕГО КОДА ===

print(f"Итого записей в датасете: {len(df)}")
print(f"Распределение классов:\n{df['label'].value_counts()}")


---

## Блок 2. Очистка датасета (5 баллов)

Сырые данные практически всегда содержат шум. Ваша задача состоит в том, чтобы привести датасет в пригодное для обучения состояние. Необходимо выполнить следующие шаги:

1. **Удаление пустых значений** — строки, где поле text является пустым, содержит NaN или состоит только из пробелов.
2. **Удаление дубликатов** — точные дубликаты по полю text.
3. **Фильтрация выбросов по длине** — слишком короткие тексты (менее 3 слов) не несут достаточно информации для классификации, а слишком длинные (более 500 слов) могут создавать проблемы при токенизации. Определите разумные пороги и удалите выбросы.


In [ ]:
# === ВАШ КОД ЗДЕСЬ ===

# 1. Удаление пустых значений
# ...

# 2. Удаление дубликатов
# ...

# 3. Фильтрация выбросов по длине
# ...

# === КОНЕЦ ВАШЕГО КОДА ===

print(f"Записей после очистки: {len(df)}")
print(f"Распределение классов:\n{df['label'].value_counts()}")


---

## Блок 3. Разделение на подвыборки и балансировка (5 баллов)

Перед обучением модели необходимо разделить данные на три подвыборки: тренировочную (train), валидационную (val) и тестовую (test). Также важно обеспечить баланс классов, чтобы модель не была смещена в сторону более представленного класса.

**Что нужно сделать:**

1. Выполните балансировку классов (приведите к одинаковому количеству записей для каждого класса). Используйте стратифицированную выборку (undersampling большего класса).
2. Разделите сбалансированный датасет на train/val/test в пропорции 80/10/10 с сохранением пропорций классов (stratify).


In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# === ВАШ КОД ЗДЕСЬ ===

# 1. Балансировка классов
# ...

# 2. Разделение на train/val/test
# ...

# === КОНЕЦ ВАШЕГО КОДА ===

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"\nРаспределение в train:\n{train_df['label'].value_counts()}")
print(f"\nРаспределение в val:\n{val_df['label'].value_counts()}")
print(f"\nРаспределение в test:\n{test_df['label'].value_counts()}")


---

## Блок 4. Количественная оценка и визуализация датасета (5 баллов)

Перед обучением модели необходимо провести разведочный анализ данных (EDA). Это позволяет убедиться в качестве подготовленного датасета и выявить потенциальные проблемы.

**Что нужно сделать:**

1. Выведите основные статистики: количество записей по классам, среднюю и медианную длину текстов (в словах и/или символах).
2. Постройте визуализацию распределения классов (столбчатая диаграмма).
3. Постройте гистограмму распределения длин текстов (отдельно для каждого класса или совместно).
4. Сделайте вывод о сбалансированности и качестве датасета.


In [ ]:
# === ВАШ КОД ЗДЕСЬ ===

# 1. Основные статистики
# ...

# 2. Визуализация распределения классов
# ...

# 3. Гистограмма длин текстов
# ...

# 4. Вывод о качестве датасета (напишите в markdown-ячейке ниже)
# ...

# === КОНЕЦ ВАШЕГО КОДА ===


---

## Блок 5. Публикация датасета на Hugging Face (5 баллов)

Опубликуйте подготовленный датасет на платформе Hugging Face Hub. Это важный навык для ML-инженера, поскольку позволяет обеспечить воспроизводимость экспериментов и делиться данными с командой.

**Что нужно сделать:**

1. Авторизуйтесь в Hugging Face Hub.
2. Загрузите датасет (train/val/test splits).
3. Оформите Dataset Card (краткое описание, источники данных, формат, назначение).


In [ ]:
from huggingface_hub import login, HfApi
from datasets import Dataset, DatasetDict

# Авторизация (вставьте ваш токен)
login(token="ваш_hf_токен")

# === ВАШ КОД ЗДЕСЬ ===

# 1. Создание DatasetDict из ваших DataFrame
# ...

# 2. Загрузка на Hub
# ...

# === КОНЕЦ ВАШЕГО КОДА ===


---

## Блок 6. Дообучение модели (код предоставлен, 0 баллов)

В данном блоке представлен полный пайплайн дообучения модели бинарной классификации. Код не требует модификации. Ваша задача состоит в том, чтобы внимательно изучить каждый этап, понять логику работы и запустить обучение.

**Что здесь происходит (объяснение):**

Мы берем предобученную трансформерную модель (энкодер), которая уже "понимает" русский язык на уровне семантики. Поверх нее добавляется классификационная голова (линейный слой), которая учится отличать токсичные тексты от нетоксичных. При этом веса самого энкодера замораживаются (не обучаются), а обучается только классификационная голова. Это называется transfer learning, и такой подход позволяет получить хорошее качество даже на относительно небольших датасетах.

Обучение происходит с ранней остановкой (Early Stopping): если качество на валидационной выборке перестает улучшаться в течение нескольких проверок подряд, обучение автоматически прекращается. Это защищает от переобучения.

### 6.1 Подготовка к обучению


In [ ]:
import torch
import gc

print(torch.__version__)
print("Доступна ли Nvidia CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Версия Nvidia CUDA:", torch.version.cuda)
    print("Используемое устройство:", torch.cuda.get_device_properties(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### 6.2 Параметры обучения


In [ ]:
# Название предобученной модели из Hugging Face Hub
MODEL_NAME = "ai-forever/ru-en-RoSBERTa"

# Максимальная длина входной последовательности в токенах
INPUT_MAX_TOKENS = 128

# Размер батча (уменьшите, если не хватает видеопамяти)
BATCH_SIZE = 128

# Максимальное количество эпох обучения
MAX_EPOCHS = 10

# Скорость обучения
LR = 2e-4

# Регуляризация весов
WEIGHT_DECAY = 0.01

# Вероятность Dropout (для предотвращения переобучения)
DROPOUT = 0.2

# Как часто проводить валидацию (каждые N шагов)
VAL_CHECK_INTERVAL = 512

# Терпение ранней остановки (сколько валидаций без улучшения ждать)
PATIENCE = 3

# Директория для сохранения модели
MODEL_SAVE_DIR = "./model"


### 6.3 Загрузка токенизатора


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Токенизатор загружен: {MODEL_NAME}")
print(f"Размер словаря: {tokenizer.vocab_size}")


### 6.4 Подготовка DataLoader


In [ ]:
from torch.utils.data import Dataset as TorchDataset, DataLoader

class TextClassificationDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Создание датасетов
train_dataset = TextClassificationDataset(
    train_df['text'].values, train_df['label'].values, tokenizer, INPUT_MAX_TOKENS
)
val_dataset = TextClassificationDataset(
    val_df['text'].values, val_df['label'].values, tokenizer, INPUT_MAX_TOKENS
)
test_dataset = TextClassificationDataset(
    test_df['text'].values, test_df['label'].values, tokenizer, INPUT_MAX_TOKENS
)

# Создание DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


### 6.5 Модель классификации


In [ ]:
import pytorch_lightning as pl
from transformers import AutoModelForSequenceClassification
from torchmetrics import Accuracy, F1Score
from torchmetrics.classification import BinaryMatthewsCorrCoef, BinaryStatScores


class EncoderClassifier(pl.LightningModule):
    def __init__(self, model_name, tokenizer, weight_class_1=1.0, dropout_rate=0.2, lr=2e-4, weight_decay=0.01):
        super().__init__()
        self.save_hyperparameters(ignore=['tokenizer'])

        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=2,
            hidden_dropout_prob=dropout_rate,
            attention_probs_dropout_prob=dropout_rate,
            classifier_dropout=dropout_rate,
            return_dict=True
        )
        self.tokenizer = tokenizer
        self.class_weights = torch.tensor([1.0, weight_class_1])
        self.lr = lr
        self.weight_decay = weight_decay

        # Метрики
        self.train_accuracy = Accuracy(task='binary')
        self.train_f1 = F1Score(task='binary')
        self.val_accuracy = Accuracy(task='binary')
        self.val_f1 = F1Score(task='binary')
        self.train_mcc = BinaryMatthewsCorrCoef()
        self.val_mcc = BinaryMatthewsCorrCoef()
        self.train_stat_scores = BinaryStatScores()
        self.val_stat_scores = BinaryStatScores()
        self.validation_count = 0

        # Замораживаем все слои кроме классификатора
        for name, param in self.model.named_parameters():
            if 'classifier' not in name:
                param.requires_grad = False
            else:
                param.requires_grad = True

        print(f"Модель: {model_name}")
        print(f"Обучаемые параметры:")
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                print(f"  {name}")

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

    def training_step(self, batch, batch_idx):
        outputs = self.forward(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['labels']
        )
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(self.device))
        loss = loss_fct(logits.view(-1, 2), batch['labels'].view(-1))

        if batch_idx % 50 == 0:
            current_lr = self.trainer.optimizers[0].param_groups[0]['lr']
            print(f"Шаг {self.global_step}: Loss = {loss.item():.6f}, LR = {current_lr:.6f}")

        with torch.no_grad():
            pred_labels = torch.argmax(logits, dim=1).float()
            true_labels = batch['labels'].int()
            self.train_accuracy(pred_labels, true_labels)
            self.train_f1(pred_labels, true_labels)
            self.train_mcc(pred_labels, true_labels)

        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_acc', self.train_accuracy, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_f1', self.train_f1, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        outputs = self.forward(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['labels']
        )
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(self.device))
        loss = loss_fct(logits.view(-1, 2), batch['labels'].view(-1))

        pred_labels = torch.argmax(logits, dim=1).float()
        true_labels = batch['labels'].int()
        self.val_accuracy(pred_labels, true_labels)
        self.val_f1(pred_labels, true_labels)
        self.val_mcc(pred_labels, true_labels)

        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_acc', self.val_accuracy, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_f1', self.val_f1, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_mcc', self.val_mcc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, self.parameters()),
            lr=self.lr,
            weight_decay=self.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=len(self.trainer.datamodule.train_dataloader()) if hasattr(self.trainer, 'datamodule') else 1000
        )
        return [optimizer], [{"scheduler": scheduler, "interval": "step"}]


### 6.6 Запуск обучения


In [ ]:
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

# Инициализация модели
model = EncoderClassifier(
    model_name=MODEL_NAME,
    tokenizer=tokenizer,
    weight_class_1=1.0,
    dropout_rate=DROPOUT,
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

# Callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=PATIENCE,
    mode='min',
    verbose=True
)

checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    dirpath=MODEL_SAVE_DIR,
    filename='best-checkpoint',
    save_top_k=1,
    mode='min'
)

# Trainer
trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='auto',
    devices=1,
    callbacks=[early_stopping, checkpoint_callback],
    val_check_interval=VAL_CHECK_INTERVAL,
    check_val_every_n_epoch=None,
    enable_progress_bar=True,
)

# Обучение
trainer.fit(model, train_loader, val_loader)

# Сохранение модели
model.model.save_pretrained(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)
print(f"\nМодель сохранена в: {MODEL_SAVE_DIR}")


---

## Блок 7. Оценка модели и визуализация ошибок (10 баллов)

После завершения обучения необходимо оценить качество модели на тестовой выборке и визуализировать результаты. Это критически важный этап, поскольку именно здесь мы понимаем, насколько модель пригодна для продакшена.

**Что нужно сделать:**

1. Вычислите метрики на тестовой выборке: Accuracy, F1, Precision, Recall, ROC AUC, MCC.
2. Постройте ROC-кривую.
3. Задайте порог классификации (threshold) и визуализируйте распределение вероятностей для ложноположительных (FP) и ложноотрицательных (FN) предсказаний. Это поможет понять, при каких уровнях уверенности модель ошибается.

**Подсказка:** Функция evaluate_model ниже уже реализована. Вам нужно ее вызвать и построить визуализации.


In [ ]:
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix, matthews_corrcoef
from tqdm import tqdm

@torch.no_grad()
def evaluate_model(model, dataloader, device=DEVICE):
    """Полная оценка модели с сохранением вероятностей"""
    model.eval()
    model = model.to(device)

    all_preds, all_labels, all_probs = [], [], []

    for batch in tqdm(dataloader, desc="Оценка модели"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)[:, 1]
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_probs.extend(probs.cpu().tolist())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    accuracy = (all_preds == all_labels).mean()
    mcc = matthews_corrcoef(all_labels, all_preds)
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)

    results = {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "mcc": float(mcc),
        "roc_auc": float(roc_auc),
        "confusion_matrix": confusion_matrix(all_labels, all_preds).tolist(),
    }

    return results, all_preds, all_labels, all_probs


In [ ]:
# === ВАШ КОД ЗДЕСЬ ===

# 1. Вызовите evaluate_model и выведите метрики
# ...

# 2. Постройте ROC-кривую
# ...

# 3. Задайте порог (например, 0.5) и визуализируйте:
#    - Распределение вероятностей для FP (модель сказала "токсично", а на самом деле нет)
#    - Распределение вероятностей для FN (модель сказала "нетоксично", а на самом деле токсично)
# ...

# === КОНЕЦ ВАШЕГО КОДА ===


---

## Блок 8. Ручной инференс и замер времени GPU vs CPU (5 баллов)

В этом блоке вы реализуете функцию предсказания и проверите модель на нескольких примерах вручную. Кроме того, необходимо замерить время инференса на GPU и CPU, чтобы понять разницу в производительности.

**Что нужно сделать:**

1. Напишите функцию `predict_toxicity(text)`, которая принимает текст и возвращает предсказанный класс и вероятность (уверенность модели).
2. Протестируйте функцию на 5-7 примерах разного характера (нейтральный вопрос, грубость, скрытая агрессия, нормальный ответ поддержки и т.д.).
3. Замерьте время классификации одного и того же текста на GPU и на CPU. Используйте `time.time()` или `torch.cuda.Event` для точного замера.


In [ ]:
import time

# === ВАШ КОД ЗДЕСЬ ===

# 1. Функция predict_toxicity(text, device)
# Должна возвращать: предсказанный класс (0 или 1), вероятность токсичности
# ...

# 2. Тестовые примеры
test_texts = [
    "Здравствуйте, чем могу помочь?",
    "Вы что, совсем тупой? Я же уже объяснял!",
    "Пожалуйста, попробуйте перезагрузить устройство",
    "Да пошел ты, надоел уже со своими вопросами",
    "Ваш запрос принят, ожидайте ответа в течение 24 часов",
    # Добавьте свои примеры
]

# 3. Замер времени на GPU
# ...

# 4. Замер времени на CPU
# ...

# 5. Сравнение результатов
# ...

# === КОНЕЦ ВАШЕГО КОДА ===


---

## Блок 9. Публикация модели на Hugging Face (5 баллов)

Финальный шаг: публикация обученной модели на Hugging Face Hub. Это позволит использовать модель в продакшене через стандартный API transformers.

**Что нужно сделать:**

1. Загрузите сохраненные веса модели и токенизатор на Hugging Face Hub.
2. Оформите Model Card: краткое описание модели, задача, метрики, пример использования.


In [ ]:
from huggingface_hub import HfApi

# === ВАШ КОД ЗДЕСЬ ===

# 1. Загрузка модели на Hub
# ...

# 2. Model Card (создайте README.md в директории модели перед загрузкой)
# ...

# === КОНЕЦ ВАШЕГО КОДА ===


---

## Критерии оценки

Для получения максимального балла за каждый блок убедитесь, что:

- Код работает без ошибок при последовательном запуске всех ячеек.
- Визуализации информативны и имеют подписи осей.
- Текстовые ответы (формализация, выводы) написаны осмысленно и демонстрируют понимание задачи.
- Датасет и модель на Hugging Face имеют оформленные карточки (Card).
- Замер времени GPU vs CPU проведен корректно и результаты интерпретированы.
